In [1]:
#Install psycopg2_binary
!pip install psycopg2_binary

In [2]:
#Import numpy,pandas, urllib.parse, and create_engine from sqlalchemy
import numpy as np
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine

In [3]:
#Credentials for postgreSQL 4 connection
DB_USER = "postgres"
DB_PASS = "***********"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"
DB_NAME = "long_covid_sdoh"

#Initaite urllb due to special character issues
encoded_pass = urllib.parse.quote_plus(DB_PASS)

# Create SQLAlchemy engine
engine = create_engine(f"postgresql://{DB_USER}:{encoded_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


In [4]:
#Cleaning fips codes to ensure 5-digit strings
def clean_fips(val):
    if pd.isna(val) or val == "" or val == -99:
        return None
    try:
        return str(int(float(val))).zfill(5)
    except (ValueError, TypeError):
        return None

In [5]:
#Ensuring Null values in SQL for survey non-responses
def clean_placeholders(df, non_response_values=[-99, -88, -999, "-99"]):
    return df.replace(non_response_values, np.nan)

In [6]:
#Updating raw data responses to corresponding data dictionary definitions for age
AGE_MAP = {
    1: "18-29",
    2: "30-39",
    3: "40-49",
    4: "50-59",
    5: "60-69",
    6: "70-79",
    7: "80+",
}

In [7]:
#Updating raw data responses to corresponding data dictionary definitions for sex
SEX_MAP = {1: "Male", 2: "Female"}

In [8]:
#Updating raw data responses to corresponding data dictionary definitions for race
RACE_MAP = {
    1: "White alone, non-Hispanic",
    2: "Black alone, non-Hispanic",
    3: "Asian alone, non-Hispanic",
    4: "Hispanic or Latino",
    5: "Two or more races / Other",
}

In [9]:
#Updating raw data responses to corresponding data dictionary definitions for income
INCOME_MAP = {
    1: "Less than $25,000",
    2: "$25,000 - $34,999",
    3: "$35,000 - $49,999",
    4: "$50,000 - $74,999",
    5: "$75,000 - $99,999",
    6: "$100,000 - $149,999",
    7: "$150,000 - $199,999",
    8: "$200,000 and above",
}

In [10]:
#Updating raw data responses to corresponding data dictionary definitions for housing
HOUSING_MAP = {1: "Own", 2: "Rent", 3: "Without payment"}

In [11]:
#Updating raw data responses to corresponding data dictionary definitions for food scarcity
FOOD_SCARCITY_MAP = {
    1: "Enough of the kinds of food wanted",
    2: "Enough, but not always the kinds of food wanted",
    3: "Sometimes not enough to eat",
    4: "Often not enough to eat",
}

In [12]:
#Updating raw data responses to corresponding data dictionary definitions for physical limitations
LIMITATION_MAP = {
    1: "No limitation",
    2: "A little limitation",
    3: "Somewhat limited",
    4: "Significant",
}

In [ ]:

# Running the etl pipeline
def run_etl():
    print("Beginning ETL Pipeline execution...")

    # Load Raw CSV Files
    raw_svi_df = pd.read_csv("data/SVI_2022_US_county.csv")
    raw_hps_df = pd.read_csv("data/pulse2023_puf_61.csv")

    # Clean non-response placeholders
    svi_df = clean_placeholders(raw_svi_df)
    hps_df = clean_placeholders(raw_hps_df)

    # -------------------------------------------------------------
    # Table 1: states
    # -------------------------------------------------------------
    print("Processing 'states' table...")
    df_states = (
        svi_df[["ST_ABBR", "STATE"]]
        .drop_duplicates()
        .rename(columns={"ST_ABBR": "state_id", "STATE": "state_name"})
    )
    df_states["census_region"] = "Unassigned"

    # -------------------------------------------------------------
    # Table 2: counties
    # -------------------------------------------------------------
    print("Processing 'counties' table...")
    svi_df["fips_code"] = svi_df["FIPS"].apply(clean_fips)

    df_counties = (
        svi_df[["fips_code", "ST_ABBR", "COUNTY", "RPL_THEMES"]]
        .rename(
            columns={
                "ST_ABBR": "state_id",
                "COUNTY": "county_name",
                "RPL_THEMES": "overall_svi_score",
            }
        )
        .dropna(subset=["fips_code"])
        .drop_duplicates(subset=["fips_code"])
    )

    # -------------------------------------------------------------
    # Table 3: patients
    # -------------------------------------------------------------
    print("Processing 'patients' table...")

    # 1. Build lookup of first valid county FIPS per state without adding extra columns to df_counties
    state_to_fips_map = (
        df_counties.groupby(df_counties["fips_code"].str[:2])["fips_code"]
        .min()
        .to_dict()
    )

    # 2. Extract numeric patient ID
    hps_df["patient_id"] = hps_df["SCRAM"].str.extract(r"(\d+)").astype(int)

    # 3. Dynamically map EST_ST to the first valid county FIPS in that state
    hps_df["fips_code"] = hps_df["EST_ST"].apply(
        lambda x: state_to_fips_map.get(str(int(x)).zfill(2)) if pd.notna(x) else None
    )

    df_patients = pd.DataFrame()
    df_patients["patient_id"] = hps_df["patient_id"]
    df_patients["fips_code"] = hps_df["fips_code"]
    df_patients["age_group"] = hps_df["TBIRTH_YEAR"].apply(
        lambda yr: (
            "18-29"
            if yr >= 1994
            else "30-39"
            if yr >= 1984
            else "40-49"
            if yr >= 1974
            else "50-59"
            if yr >= 1964
            else "60-69"
            if yr >= 1954
            else "70+"
        )
    )
    df_patients["biological_sex"] = hps_df["EGENID_BIRTH"].map(
        {1: "Male", 2: "Female"}
    )
    df_patients["race_ethnicity"] = hps_df["RRACE"].map(RACE_MAP)
    df_patients = df_patients.drop_duplicates(subset=["patient_id"])

    # Filter patients to match existing county FIPS codes
    valid_fips = set(df_counties["fips_code"])
    df_patients = df_patients[df_patients["fips_code"].isin(valid_fips)]
    valid_patient_ids = set(df_patients["patient_id"])

    # -------------------------------------------------------------
    # Table 4: patient_economics
    # -------------------------------------------------------------
    print("Processing 'patient_economics' table...")
    df_economics = pd.DataFrame()
    df_economics["patient_id"] = hps_df["patient_id"]
    df_economics["income_bracket"] = hps_df["INCOME"].map(INCOME_MAP)
    df_economics["employment_status"] = hps_df["ANYWORK"].map(
        {1: "Employed", 2: "Unemployed"}
    )
    df_economics["education_level"] = hps_df["EEDUC"].map(
        {
            1: "Less than High School",
            2: "Some High School",
            3: "High School Diploma/GED",
            4: "Some College",
            5: "Associate Degree",
            6: "Bachelor Degree",
            7: "Graduate Degree",
        }
    )
    df_economics["health_insurance_type"] = hps_df["PRIVHLTH"].map(
        {1: "Employer/Private", 2: "Uninsured"}
    )

    df_economics = df_economics[
        df_economics["patient_id"].isin(valid_patient_ids)
    ].drop_duplicates(subset=["patient_id"])

    # -------------------------------------------------------------
    # Table 5: patient_housing
    # -------------------------------------------------------------
    print("Processing 'patient_housing' table...")
    df_housing = pd.DataFrame()
    df_housing["patient_id"] = hps_df["patient_id"]
    df_housing["housing_tenure"] = hps_df["TENURE"].map(HOUSING_MAP)
    df_housing["eviction_risk_flag"] = (
        hps_df["EVICT"].map({1: True, 2: False}).astype("boolean")
    )
    df_housing["missed_rent_payment"] = (
        hps_df["RENTCUR"].map({1: False, 2: True}).astype("boolean")
    )

    df_housing = df_housing[
        df_housing["patient_id"].isin(valid_patient_ids)
    ].drop_duplicates(subset=["patient_id"])

    # -------------------------------------------------------------
    # Table 6: patient_food_security
    # -------------------------------------------------------------
    print("Processing 'patient_food_security' table...")
    df_food = pd.DataFrame()
    df_food["patient_id"] = hps_df["patient_id"]
    df_food["food_scarcity_status"] = hps_df["CURFOODSUF"].map(
        FOOD_SCARCITY_MAP
    )
    df_food["used_food_assistance"] = (
        hps_df["FREEFOOD"].map({1: True, 2: False}).astype("boolean")
    )

    df_food = df_food[
        df_food["patient_id"].isin(valid_patient_ids)
    ].drop_duplicates(subset=["patient_id"])

    # -------------------------------------------------------------
    # Table 7: surveys_taken
    # -------------------------------------------------------------
    print("Processing 'surveys_taken' table...")
    df_surveys = pd.DataFrame()
    df_surveys["patient_id"] = hps_df["patient_id"]
    df_surveys["survey_phase"] = hps_df["WEEK"].apply(
        lambda x: f"Phase {int(x)}" if pd.notna(x) else "Phase 61"
    )
    df_surveys["collection_date"] = pd.to_datetime("2023-06-01")

    df_surveys = (
        df_surveys[df_surveys["patient_id"].isin(valid_patient_ids)]
        .reset_index(drop=True)
    )
    df_surveys["survey_response_id"] = range(1, len(df_surveys) + 1)

    df_surveys = df_surveys[[
        "survey_response_id",
        "patient_id",
        "survey_phase",
        "collection_date",
    ]]

    # -------------------------------------------------------------
    # Table 8: long_covid_cases
    # -------------------------------------------------------------
    print("Processing 'long_covid_cases' table...")
    hps_valid = hps_df[hps_df["patient_id"].isin(valid_patient_ids)].reset_index(
        drop=True
    )

    df_covid = pd.DataFrame()
    df_covid["case_id"] = range(1, len(df_surveys) + 1)
    df_covid["survey_response_id"] = df_surveys["survey_response_id"]
    df_covid["had_acute_covid"] = (
        hps_valid["HADCOVIDRV"].map({1: True, 2: False}).astype("boolean")
    )
    df_covid["long_covid_flag"] = (
        hps_valid["LONGCOVID"].map({1: True, 2: False}).astype("boolean")
    )

    has_long_covid = df_covid["long_covid_flag"].fillna(False)
    df_covid["symptom_duration_months"] = np.where(has_long_covid, 3, 0)
    df_covid["activity_limitation_level"] = hps_valid["SYMPTMIMPCT"].map(
        LIMITATION_MAP
    )

    tables_in_order = [
        ("states", df_states),
        ("counties", df_counties),
        ("patients", df_patients),
        ("patient_economics", df_economics),
        ("patient_housing", df_housing),
        ("patient_food_security", df_food),
        ("surveys_taken", df_surveys),
        ("long_covid_cases", df_covid),
    ]

    print("\nInserting data into PostgreSQL database...")
    with engine.begin() as connection:
        for table_name, df in tables_in_order:
            print(f"-> Uploading {len(df)} rows to '{table_name}'...")
            df.to_sql(
                name=table_name,
                con=connection,
                if_exists="append",
                index=False,
                chunksize=1000,
            )

    print("\nETL Pipeline completed successfully!")


# Execute function directly
run_etl()

Beginning ETL Pipeline execution...
Processing 'states' table...
Processing 'counties' table...
Processing 'patients' table...
Processing 'patient_economics' table...
Processing 'patient_housing' table...
Processing 'patient_food_security' table...
Processing 'surveys_taken' table...
Processing 'long_covid_cases' table...

Inserting data into PostgreSQL database...
-> Uploading 51 rows to 'states'...
-> Uploading 3144 rows to 'counties'...
-> Uploading 68454 rows to 'patients'...
-> Uploading 68454 rows to 'patient_economics'...
-> Uploading 68454 rows to 'patient_housing'...
-> Uploading 68454 rows to 'patient_food_security'...
-> Uploading 68454 rows to 'surveys_taken'...
-> Uploading 68454 rows to 'long_covid_cases'...

ETL Pipeline completed successfully!
